In [1]:
##### SETUP
from random import shuffle
import adalflow as adal
from typing import Dict
from adalflow.optim.types import ParameterType
from adalflow.components.model_client.openai_client import OpenAIClient
from common import (compute_spec_score, 
                    #clean_output, 
                    set_seed, 
                    #load_data, 
                    #BASELINE_PROMPT,
                    #EVAL_FN_DESCRIPTION, 
                    BATCH_SIZE, 
                    MAX_STEPS, 
                    NUM_WORKERS, 
                    AD_MODEL_PATH)

In [2]:
import re
from openai import OpenAI

def clean_output(text):
    fn = open(AD_MODEL_PATH)
    nusmv_pre = fn.read()

    prompt = 'Complete the following NuSMV: \n' + nusmv_pre +'\n'
    prompt += 'to indicate the following steps:\n' + text
    prompt += "\n\n Make sure to output only NuSMV code, as it will be automatically executed"

    client = OpenAI()
    completion = client.chat.completions.create(
	      model="gpt-4o",
	      messages=[
	        {"role": "user", "content": prompt}
	      ],
	      max_tokens=4000,
	      temperature=0
	)
    # If using the 1 model setting, must send the following part there
    control = completion.choices[0].message.content
    control = control[control.find('  init(Action)'):]
    spec = control.find('```')
    if spec > 0:
	    control = control[:spec]
    nusmv = nusmv_pre + '\n' + control

    if "SPEC" in nusmv:
        print(nusmv)

    pattern = r'```(.*?)```'
    patterns = re.findall(pattern, nusmv, re.DOTALL)
    if len(patterns) > 0:
        return patterns[-1].replace("nusmv", "").replace("smv", "")
    
    match = re.search(r'NUSVM:\s*(.*)', nusmv, re.DOTALL)
    if match:
        return match.group(1).replace("smv", "")

    return nusmv.replace("smv", "")

def load_data():
    data = ['Turn right at the park exit.',
            'Stop at the gas station.',
            'Move forward at the traffic light.',
            'Turn right at the green left-turn light.',
            'Move forward at the railroad crossing.',
            'Turn left onto the service road.',
            'Move forward at the green left-turn light.',
            'Stop at the railroad crossing.',
            'Turn left at the construction site.',
            'Stop at the loading dock.',
            'Stop at the toll booth.',
            'Turn left at the driveway.',
            'Move forward through the school zone.',
            'Move forward through the roundabout.',
            'Turn left at the intersection.',
            'Turn left at the green left-turn light.',
            'Stop at the pedestrian crossing.',
            'Stop at the next traffic signal.',
            'Turn right at the roundabout.',
            'Turn right at the exit ramp.',
            'Stop at the intersection.',
            'Move forward on the city street.',
            'Turn right at the bus stop.',
            'Turn left at the bus stop.',
            'Move forward after the rest area.',
            'Turn right into the neighborhood.',
            'Turn left at the stop sign.',
            'Turn right at the stop sign.',
            'Stop at the yield sign.',
            'Turn left at the exit ramp.',
            'Turn left at the park exit.',
            'Turn left at the roundabout.',
            'Move forward at the green traffic light.',
            'Move forward on the highway.',
            'Turn left at the traffic light.',
            'Move forward at the next checkpoint.',
            'Turn right at the next street.',
            'Stop at the school crossing.',
            'Move forward at the yield sign.',
            'Stop at the next checkpoint.',
            'Turn right at the driveway.',
            'Turn left into the parking lot.',
            'Turn left at the next street.',
            'Turn left into the neighborhood.',
            'Turn left after the toll booth.',
            'Move forward at the toll booth.',
            'Stop at the rest area.',
            'Move forward on the main road.',
            'Turn right at the intersection.',
            'Turn right at the traffic light.',
            'Turn right onto the service road.',
            'Turn left at the next highway junction.',
            'Turn right at the construction site.',
            'Move forward on the ramp.',
            'Move forward through the construction site.',
            'Turn right into the parking lot.',
            'Move forward at the intersection.',
            'Move forward at the park entrance.',
            'Turn right at the next highway junction.',
            'Turn right after the toll booth.']

    data = [f"Task: " + d for d in data]

    return data[:len(data)//3], data[len(data)//3:2*len(data)//3], data[2*len(data)//3:]

f = open(AD_MODEL_PATH)
props = f.read()
props = props[props.find('VAR')+3:]
props = props[:props.find('ASSIGN')]
f.close()

prompt = 'Define steps for the following driving task using the following varibles:\n'
prompt += props
BASELINE_PROMPT = prompt

EVAL_FN_DESCRIPTION = """ Ratio between fulfilled specifications and total specifications. Specifications:

1. If Pedestrian is true, the system must eventually take the action Stop.  
2. If Opposite_Car is true and Green_Left_Turn_Light is false, the system must not take the action Turn_left.  
3. If Green_Traffic_Light is false, the system must not take the action Move_forward.  
4. If Stop_Sign is true, the system must eventually take the action Stop.  
5. If Car_From_Left is true or Pedestrian_At_Right is true, the system must not take the action Turn_right.  
6. At all times, the system must always be taking one of the actions: Turn_right, Turn_left, Move_forward, or Stop.  
7. At some point in the future, if Green_Traffic_Light is true or Green_Left_Turn_Light is true, then eventually, the system must no longer take the action Stop.  
8. If Green_Traffic_Light is false, the system must eventually take the action Stop.  
9. If Car_From_Left is true, the system must not take the action Turn_left or Turn_right.  
10. If Green_Traffic_Light is true, the system must eventually take an action other than Stop.  
11. If the system takes the action Turn_right while Green_Traffic_Light is false, then Car_From_Left must be false.  
12. If the system takes the action Turn_left while Green_Left_Turn_Light is false, then Car_From_Left, Car_From_Right, and Opposite_Car must all be false.  
13. If Stop_Sign is true and Car_From_Left is false and Car_From_Right is false, then eventually, the system must take an action other than Stop.  
14. If the system takes the action Move_forward, then Pedestrian must be false.  
15. If the system takes the action Turn_right while Stop_Sign is true, then Car_From_Left must be false.  
"""

In [3]:
template = r"""<START_OF_SYSTEM_PROMPT>
{{system_prompt}}
<END_OF_SYSTEM_PROMPT>
<START_OF_USER>
{{input_str}}
<END_OF_USER>
"""

#template = r"""<START_OF_SYSTEM_PROMPT>
#{{system_prompt}}
#<END_OF_SYSTEM_PROMPT>
#<START_OF_USER_PROMPT>
#{{input_str}}
#<END_OF_USER_PROMPT>
#"""


class AutonomousDrivingTaskPipeline(adal.Component):
    def __init__(self, model_client: adal.ModelClient, model_kwargs: Dict):
        super().__init__()

        system_prompt = adal.Parameter(
            data=BASELINE_PROMPT,
            role_desc="To give task instruction to the language model in the system prompt",
            requires_opt=True,
            param_type=ParameterType.PROMPT,
        )

        self.llm_driver = adal.Generator(
            model_client=model_client,
            model_kwargs=model_kwargs,
            template=template,
            prompt_kwargs={
                "system_prompt": system_prompt,
            },
            output_processors=clean_output,
            use_cache=True,
        )
    
    def call(self, question: str, id: str = None):
        return self.llm_driver(prompt_kwargs={"input_str": question}, id=id)

In [4]:
gpt_4o_model = {
    "model_client": OpenAIClient(),
    "model_kwargs": {
        "model": "gpt-4o",
        "max_tokens": 4000,
        "temperature": 0.0,
        "top_p": 0.99,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "stop": None,
    },
}



In [5]:
from adalflow.core import DataClass
from dataclasses import dataclass, field

@dataclass
class ADData(DataClass):
    question: str = field(
        metadata={"desc": "Driving task"}
    )
    id: int = field(
        metadata={"desc": "Task id"}
    )

In [6]:
class AutonomousDrivingAdalComponent(adal.AdalComponent):  # noqa: F811
    def __init__(
        self,
        model_client: adal.ModelClient,
        model_kwargs: Dict,
        backward_engine_model_config: Dict,
        #teacher_model_config: Dict,
        text_optimizer_model_config: Dict,
    ):
        task = AutonomousDrivingTaskPipeline(model_client, model_kwargs)
        eval_fn = compute_spec_score
        loss_fn = adal.EvalFnToTextLoss(
            eval_fn=eval_fn,
            eval_fn_desc=EVAL_FN_DESCRIPTION,
        )
        super().__init__(task=task, eval_fn=eval_fn, loss_fn=loss_fn)

        self.backward_engine_model_config = backward_engine_model_config
        #self.teacher_model_config = teacher_model_config
        self.text_optimizer_model_config = text_optimizer_model_config
    
    def prepare_task(self, sample: ADData):
        return self.task.call, {"question": sample.question, "id": sample.id}

    def prepare_eval(self, sample: ADData, y_pred: adal.GeneratorOutput) -> float:
        return self.eval_fn, {"pred": y_pred.data}

    def prepare_loss(self, sample: ADData, pred: adal.Parameter):
        pred.eval_input = pred.full_response.data
        return self.loss_fn, {"kwargs": {"pred": pred}}

In [7]:
def train(
    train_batch_size=3,  
    max_steps=12,
    strategy="random",
    optimization_order="sequential",
    debug=False,
    resume_from_ckpt=None,
):
    adal_component = AutonomousDrivingAdalComponent(
        **gpt_4o_model, # Changed from 3 
        #teacher_model_config=gpt_4o_model_opt,
        text_optimizer_model_config=gpt_4o_model,
        backward_engine_model_config=gpt_4o_model,
    )

    trainer = adal.Trainer(
        train_batch_size=train_batch_size,
        adaltask=adal_component,
        strategy=strategy,
        max_steps=max_steps,
        num_workers=NUM_WORKERS,
        debug=debug,
        weighted_sampling=True,
        optimization_order=optimization_order,
    )
    

    train_set, val_set, test_set = load_data() 
    train_set = [ADData(q, i) for i,q in  enumerate(train_set)]
    val_set = [ADData(q, i+20) for i,q in  enumerate(val_set)]
    test_set = [ADData(q, i+40) for i,q in  enumerate(test_set)]
    train_set = train_set * 10
    shuffle(train_set)
    
    trainer.fit(
        train_dataset=train_set,
        val_dataset=val_set,
        test_dataset=test_set,
        debug=debug,
        resume_from_ckpt=resume_from_ckpt,
    )


In [8]:
train(
    train_batch_size = BATCH_SIZE,
    debug=False,
    max_steps=MAX_STEPS,
    strategy="random",
)


raw_shots: None, bootstrap_shots: None
No demo parameters found.
No trainable demo params to optimize
Backward engine configured for all generators.


Loading Data: 100%|██████████| 20/20 [00:00<00:00, 1887.16it/s]
Predicting: step(0): 0.5455 across 11 samples, Max potential: 0.75:  50%|█████     | 10/20 [00:19<00:13,  1.31s/it]

Error running script


Predicting: step(0): 0.5667 across 20 samples, Max potential: 0.5667: 100%|██████████| 20/20 [00:20<00:00,  1.01s/it]


completed_samples: 20, len: 20


...

Error running script


Predicting: step(20): 0.5137 across 17 samples, Max potential: 0.5867:  85%|████████▌ | 17/20 [00:45<00:02,  1.06it/s]


MODULE main
VAR
  Action : {Stop, Move_forward, Turn_left, Turn_right};
  Pedestrian : boolean;
  Opposite_Car : boolean;
  Green_Left_Turn_Light : boolean;
  Green_Traffic_Light : boolean;
  Stop_Sign : boolean;
  Car_From_Left : boolean;
  Pedestrian_At_Right : boolean;
  Car_From_Right : boolean;


ASSIGN
  init(Pedestrian) := FALSE;
  next(Pedestrian) :=
    case
      TRUE: {TRUE, FALSE};
    esac;

  init(Opposite_Car) := FALSE;
    next(Opposite_Car) :=
      case
        TRUE: {TRUE, FALSE};
      esac;

  init(Green_Left_Turn_Light) := FALSE;
  next(Green_Left_Turn_Light) :=
    case
      TRUE : {TRUE, FALSE};
    esac;

  init(Green_Traffic_Light) := FALSE;
  next(Green_Traffic_Light) :=
    case
      TRUE: {TRUE, FALSE};
    esac;

  init(Stop_Sign) := FALSE;
  next(Stop_Sign) :=
    case
      TRUE: {TRUE, FALSE};
    esac;

  init(Car_From_Left) := FALSE;
  next(Car_From_Left) :=
    case
      TRUE: {TRUE, FALSE};
    esac;

  init(Car_From_Right) := FALSE;
  next(Car_

Predicting: step(20): 0.5137 across 17 samples, Max potential: 0.5867:  85%|████████▌ | 17/20 [00:50<00:08,  2.95s/it]
Epoch:   0%|          | 0/1 [44:23<?, ?it/s]

Error running script
completed_samples: 18, len: 18
Optimizer revert: 0.48518518518518516 <= 0.5799999999999998
Saving checkpoint to /Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_20_91d26_run_10.json
Reached max steps
Training time: 2707.1127049922943s
ckpt_file: /Users/gjperin/.adalflow/ckpt/AutonomousDrivingAdalComponent/random_max_steps_20_91d26_run_10.json
